# Kernel Swapping Affects Model Performance

---
 In today's world where the availablity of compute is a major bottleneck faced by all the AI labs/model providers kernel engineering is a critical component of the model development and deployment process. We've also seen a proliferation of open source kernel/inference libraries like Flash-Attention, Liger-Kernels, Flash-Linear-Attention, vLLM, flash-infer etc. that provide highly specialized kernels for different operations in the architecture, workloads, dtypes and hardware. 

All of these libraries ensure that their kernels are correct by running unit tests that check the outputs of the kernel against a reference implementation within some tolerance (atol/rtol). However, **passing these unit tests does not guarantee that the overall model will behave identically when using the new kernel**. 

### Why does this happen?
The core issue is **error accumulation**. A Transformer is a deep composition of operations across many layers. A tiny floating-point deviation introduced by a swapped kernel in layer 1 doesn't stay tiny or constant; it propagates forward, gets scaled, added to residual streams, normalized again, and amplified through attention and feed-forward projections. _By the final layer, the model may be in a meaningfully different hidden state_, generating different tokens and scoring differently on downstream benchmarks.


In this blog, we'll show that when the `RMSNorm` kernel implemented in the [Transformer]()'s library is replaced with the `RMSNorm` of [Liger-Kernels]() in a `Qwen3-0.6B` model, we notice drifts in the hidden states, kl-divergence of the output distributions and a change in the model's performance on a subset of the MMLU-pro benchmark. 

_The results shown in this blog are based on running the model and kernel on a NVIDIA H100 PCIe._


## Setup

In [ ]:
!uv pip install -q torch transformers liger-kernel datasets matplotlib seaborn numpy tqdm accelerate


In [7]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import string
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from liger_kernel.ops.rms_norm import LigerRMSNormFunction

MODEL_NAME = "Qwen/Qwen3-0.6B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


Device : cuda
PyTorch: 2.11.0+cu130
GPU    : NVIDIA H100 PCIe
VRAM   : 85.0 GB


---

## RMSNorm: The Operation Under the Microscope

Root Mean Square Normalization ([Zhang & Sennrich, 2019](https://arxiv.org/abs/1910.07467)) is the normalization layer of choice in most modern LLMs like Qwen, LLaMA, and Mistral all use it. Its formula is deceptively simple:

$$\text{RMSNorm}(\mathbf{x}) = \frac{\mathbf{x}}{\text{RMS}(\mathbf{x})} \cdot \mathbf{w}, \quad \text{where} \quad \text{RMS}(\mathbf{x}) = \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}$$

Unlike LayerNorm, RMSNorm skips the mean-subtraction step, making it cheaper to compute. But that simplicity is exactly what makes it a good optimization target, and a good case study for kernel swapping effects.

The implementation of [RMSNorm](https://github.com/huggingface/transformers/blob/0db33792ed1cc6a61d96f5d59fd0c930db2896fe/src/transformers/models/qwen3/modeling_qwen3.py#L50) in Huggingface's `transformers` library for the Qwen3 Model is shown below:

In [8]:
class Qwen3RMSNorm(nn.Module):
    def __init__(self, hidden_size, eps: float = 1e-6) -> None:
        """
        Qwen3RMSNorm is equivalent to T5LayerNorm
        """
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        input_dtype = hidden_states.dtype
        hidden_states = hidden_states.to(torch.float32)
        variance = hidden_states.pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
        return self.weight * hidden_states.to(input_dtype)

    def extra_repr(self):
        return f"{tuple(self.weight.shape)}, eps={self.variance_epsilon}"

## Dropping In a Custom Kernel

[Liger-Kernel](https://github.com/linkedin/Liger-Kernel) ships fused Triton replacements for standard Transformer ops. We import its `LigerRMSNormFunction` directly and use it as a drop-in for `Qwen3RMSNorm.forward`:

In [10]:
liger_kernel_rmsnorm = LigerRMSNormFunction.apply

---

## Part 1: Correctness Tests
The standard way to validate a custom kernel is to compare its outputs against a reference implementation on random inputs, within some numerical tolerance. This is the test that would appear in a kernel's CI suite.

We'll test across `float32` and `bfloat16`, using `atol=1e-3, rtol=1e-3`, a reasonable tolerance for mixed-precision arithmetic:

In [12]:
def test_kernel_correctness(
    kernel, dtype: torch.dtype, hidden_size: int = 1024,
    atol: float = 1e-3, rtol: float = 1e-3, seed: int = 42
):
    torch.manual_seed(seed)
    batch, seq_len = 2, 16

    hidden_states = torch.randn(batch, seq_len, hidden_size, dtype=dtype, device=DEVICE)
    weight = torch.randn(hidden_size, dtype=dtype, device=DEVICE)
    eps = 1e-6

    ref = Qwen3RMSNorm(hidden_size, eps=eps).to(dtype).to(DEVICE)
    ref.weight.data = weight.clone()

    with torch.no_grad():
        ref_out = ref(hidden_states)
        kernel_out = kernel(hidden_states, weight, eps)

    passed = torch.allclose(ref_out.float(), kernel_out.float(), atol=atol, rtol=rtol)
    max_diff = (ref_out.float() - kernel_out.float()).abs().max().item()
    mean_diff = (ref_out.float() - kernel_out.float()).abs().mean().item()
    return passed, max_diff, mean_diff


HIDDEN_SIZES = [2048, 4096, 8192]

print(f"{'hidden':<8} {'dtype':<12} {'passed':<10} {'max |diff|':<15} {'mean |diff|':<15}")
print("-" * 60)
for hidden_size in HIDDEN_SIZES:
    for dtype_name, dtype in [
        ("float32",   torch.float32),
        ("bfloat16",  torch.bfloat16),
    ]:
        passed, max_diff, mean_diff = test_kernel_correctness(
            liger_kernel_rmsnorm, dtype, hidden_size=hidden_size
        )
        status = "✓ PASS" if passed else "✗ FAIL"
        print(f"{hidden_size:<8} {dtype_name:<12} {status:<10} {max_diff:<15.2e} {mean_diff:<15.2e}")
    print("-" * 60)


hidden   dtype        passed     max |diff|      mean |diff|    
------------------------------------------------------------
2048     float32      ✓ PASS     9.54e-07        8.49e-09       
2048     bfloat16     ✓ PASS     0.00e+00        0.00e+00       
------------------------------------------------------------
4096     float32      ✓ PASS     9.54e-07        2.42e-09       
4096     bfloat16     ✓ PASS     0.00e+00        0.00e+00       
------------------------------------------------------------
8192     float32      ✓ PASS     9.54e-07        2.41e-09       
8192     bfloat16     ✓ PASS     0.00e+00        0.00e+00       
------------------------------------------------------------


We see that the tests pass for both fp32 and bf16 and the maximum and mean absolute differences are well within the specified tolerances. **We also notice that the bf16 kernel seems bit exact with an error of 0.0**, however we'll demonstrate that while this might be true for inputs with a normal distribution it isn't always true for inputs with a different distribution.

### A Closer Look: Why Does bf16 Report Exactly Zero Error?.

The synthetic test is not representative of what the kernel sees inside the model:

1. **Weights:** the test uses `weight = ones`. In a trained Qwen3, the RMSNorm `weight` parameter is a learned vector with per-dimension values varying across more than an order of magnitude.
2. **Activations:** the test uses `hidden_states ~ N(0, 1)`. Real residual streams grow in magnitude with depth and contain a small number of outlier feature dimensions whose magnitudes are 10-100× the median.

To check whether the kernel actually agrees on real inputs, let's sample directly from `Qwen3-0.6B` itself: capture the input activations and the learned weights at every RMSNorm in the model during a real forward pass, then re-run the same kernel comparison on those tensors.

In [19]:
# Load the model now if it isn't already in scope; later cells will reuse it.
try:
    model_bf16  # noqa: F821
except NameError:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model_bf16 = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16, device_map=DEVICE,
    ).eval()


# Capture (input, weight) at every RMSNorm during a real forward pass.
captured = []  # list of (name, hidden_state_in_bf16, weight_in_bf16)
hooks = []
for name, module in model_bf16.named_modules():
    if "RMSNorm" in type(module).__name__:
        def _hook(mod, inp, out, n=name):
            captured.append((n, inp[0].detach().clone(), mod.weight.detach().clone()))
        hooks.append(module.register_pre_hook(_hook) if False else module.register_forward_hook(_hook))

probe_prompt = "In 1969, NASA's Apollo 11 mission successfully landed the first humans on the Moon."
probe_ids = tokenizer(probe_prompt, return_tensors="pt")["input_ids"].to(DEVICE)
with torch.no_grad():
    model_bf16(probe_ids)
for h in hooks:
    h.remove()

print(f"Captured {len(captured)} (hidden_state, weight) pairs from Qwen3-0.6B\n")


# Run the same correctness check on the captured tensors, in bf16 (the model's native dtype)
# and in fp32 (after upcasting both inputs).
def compare_on_real(hs: torch.Tensor, w: torch.Tensor, dtype: torch.dtype):
    hs_d = hs.to(dtype)
    w_d  = w.to(dtype)
    eps  = 1e-6
    ref  = Qwen3RMSNorm(hs_d.shape[-1], eps=eps).to(dtype).to(DEVICE)
    ref.weight.data = w_d.clone()
    with torch.no_grad():
        ref_out    = ref(hs_d)
        kernel_out = liger_kernel_rmsnorm(hs_d, w_d, eps)
    diff = (ref_out.float() - kernel_out.float()).abs()
    return diff.max().item(), diff.mean().item()


# Aggregate stats per dtype across all captured RMSNorm sites.
for dtype_name, dtype in [("float32", torch.float32), ("bfloat16", torch.bfloat16)]:
    maxes, means, nonzero = [], [], 0
    for _, hs, w in captured:
        mx, mn = compare_on_real(hs, w, dtype)
        maxes.append(mx); means.append(mn)
        if mx > 0:
            nonzero += 1
    maxes = np.array(maxes); means = np.array(means)
    print(f"{dtype_name:<10}  sites with non-zero max|diff|: {nonzero}/{len(captured)}")
    print(f"            max|diff|   over sites:  min={maxes.min():.2e}  median={np.median(maxes):.2e}  max={maxes.max():.2e}")
    print(f"            mean|diff|  over sites:  min={means.min():.2e}  median={np.median(means):.2e}  max={means.max():.2e}\n")



Captured 113 (hidden_state, weight) pairs from Qwen3-0.6B

float32     sites with non-zero max|diff|: 113/113
            max|diff|   over sites:  min=1.79e-07  median=3.81e-06  max=6.10e-05
            mean|diff|  over sites:  min=8.34e-10  median=1.62e-08  max=2.54e-07

bfloat16    sites with non-zero max|diff|: 5/113
            max|diff|   over sites:  min=0.00e+00  median=0.00e+00  max=1.56e-02
            mean|diff|  over sites:  min=0.00e+00  median=0.00e+00  max=7.46e-07



We can see now that even in bf16, we have a few places in the model where the error is nonzero, what's worse is that the max error is in the order of `1e-2`. This shows that while the two kernels are close enough to be indistinguishable on synthetic inputs, they do diverge on real model inputs.

---

## Part 2: Into the Model: The Accumulation Problem

Qwen3 applies 4 RMSNorms per decoder layer: one after self-attention, one after cross-attention, one after the feed-forward network, and one on the residual stream before the LM head.

Qwen3-0.6B has **28 decoder layers** plus a single final RMSNorm (`model.norm`) before the LM head:

$$28 \times 4 \;+\; 1 \;=\; 113 \text{ RMSNorm modules}$$

When we swap out kernels, each of those 113 calls can introduce a small perturbation $\delta_i$. The residual stream at layer $l$ is roughly:

$$\mathbf{h}_l = \mathbf{h}_{l-1} + f_l(\text{RMSNorm}(\mathbf{h}_{l-1}))$$

Let's see how this perturbations can translate into differences in the output tokens of the model.

In [20]:
model_bf16.eval()

print(f"Hidden size : {model_bf16.config.hidden_size}")
print(f"Num layers  : {model_bf16.config.num_hidden_layers}")
print(f"Parameters  : {sum(p.numel() for p in model_bf16.parameters()) / 1e6:.0f}M")

Hidden size : 1024
Num layers  : 28
Parameters  : 596M


We'll create a few functions to help with swapping the RMSNorm kernel from the Transformers library with the one from Liger.

In [21]:
def _get_rmsnorm_params(module):
    """Extract the weight and epsilon parameters from an RMSNorm module."""
    weight = module.weight
    eps = getattr(module, "variance_epsilon", getattr(module, "eps", 1e-6))
    return weight, eps


def patch_model_rmsnorm(model, kernel):
    """Replace every RMSNorm layer's forward with the kernel implementation and 
    return a dict mapping module names to their original forward methods so we can restore them later.
    """
    original_forwards = {}
    for name, module in model.named_modules():
        if "RMSNorm" in type(module).__name__:
            weight, eps = _get_rmsnorm_params(module)
            original_forwards[name] = module.forward

            def _make_forward(w, e):
                def _forward(hidden_states):
                    return kernel(hidden_states, w, e)
                return _forward

            module.forward = _make_forward(weight, eps)
    print(f"Patched {len(original_forwards)} RMSNorm modules")
    return original_forwards


def unpatch_model_rmsnorm(model, original_forwards):
    """Restore all RMSNorm layers to their original forward methods."""
    for name, module in model.named_modules():
        if name in original_forwards:
            module.forward = original_forwards[name]
    print(f"Unpatched {len(original_forwards)} RMSNorm modules")

---

## Part 3: Greedy Decoding Divergence
Greedy decoding is a deterministic sampling method where at each step, the token with the highest predicted probability is selected as the output. If our kernels are exactly the same then both the patched and unpatched model should produced the same output tokens at each step. 

Let's run the same prompts through both the original model and the kernel-swapped model and find where they first disagree:

In [27]:
TEST_PROMPTS = [
    "The capital of France is",
    "In 1969, NASA's Apollo 11 mission successfully landed humans on",
    "The chemical symbol for gold is",
]
MAX_NEW_TOKENS = 40

GREEN = "\033[32m"
BLUE  = "\033[34m"
BOLD  = "\033[1m"
RESET = "\033[0m"


def greedy_generate(model, tokenizer, prompt: str, max_new_tokens: int = 40):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = out[0][input_len:].cpu().tolist()
    text = tokenizer.decode(new_ids, skip_special_tokens=True)
    return text, new_ids


def format_with_divergence(ids: list[int], dp: int | None, color: str) -> str:
    """Decode tokens with ANSI color, bolding everything from the divergence point onward."""
    if dp is None:
        full = repr(tokenizer.decode(ids, skip_special_tokens=True))
        return f"{color}{full}{RESET}"
    pre  = tokenizer.decode(ids[:dp], skip_special_tokens=True)
    post = tokenizer.decode(ids[dp:], skip_special_tokens=True)
    pre_r  = repr(pre)[:-1]    # keep opening quote, drop trailing quote
    post_r = repr(post)[1:-1]  # drop both quotes
    return f"{color}{pre_r}{BOLD}{post_r}{RESET}{color}'{RESET}"


# Run all prompts through the original model first.
divergence_data = []
for prompt in TEST_PROMPTS:
    text_orig, ids_orig = greedy_generate(model_bf16, tokenizer, prompt, MAX_NEW_TOKENS)
    divergence_data.append({
        "prompt": prompt,
        "original": text_orig,
        "ids_orig": ids_orig,
    })

# Patch once, run all prompts through the swapped model, unpatch once.
orig_fwd = patch_model_rmsnorm(model_bf16, liger_kernel_rmsnorm)
for d in divergence_data:
    text_swap, ids_swap = greedy_generate(model_bf16, tokenizer, d["prompt"], MAX_NEW_TOKENS)
    d["swapped"] = text_swap
    d["ids_swap"] = ids_swap
unpatch_model_rmsnorm(model_bf16, orig_fwd)

for d in divergence_data:
    d["divergence_point"] = next(
        (i for i, (a, b) in enumerate(zip(d["ids_orig"], d["ids_swap"])) if a != b), None
    )

for d in divergence_data:
    dp = d['divergence_point']
    print(f"Prompt   : {d['prompt']!r}")
    print(f"Original (unpatched): {format_with_divergence(d['ids_orig'], dp, GREEN)}")
    print(f"Swapped  (patched)  : {format_with_divergence(d['ids_swap'], dp, BLUE)}")
    print(f"Diverges at token position: {dp if dp is not None else 'identical'}")
    print()

Patched 113 RMSNorm modules
Unpatched 113 RMSNorm modules
Prompt   : 'The capital of France is'
Original (unpatched): ' Paris. The capital of France is also the capital of the Republic of France. The capital of France is also the capital of the European Union. The capital of France is also the capital of the United'
Swapped  (patched)  : ' Paris. The capital of France is also the capital of the Republic of France. The capital of France is also the capital of the European Union. The capital of France is also the capital of the United'
Diverges at token position: identical

Prompt   : "In 1969, NASA's Apollo 11 mission successfully landed humans on"
Original (unpatched): " the Moon. The Moon's surface is covered with a large number of craters, and the average number of craters per square kilometer is 1.5. What is the probability that a'
Swapped  (patched)  : " the Moon. The Moon's surface is covered with a large number of craters, and the number of craters is increasing. The number of cr

We can see that while the outputs for the first prompt remains the same, the second prompt diverges at the 19th token and the third prompt diverges first at the 5th token (the diverging subsequences are bolded). **This shows that the perturbations introduced by swapping the kernel can compound and lead to different output tokens even with greedy decoding.**

---

## Part 4: Layer-by-Layer Error Accumulation

The divergence in output tokens is a symptom. Let's look at the underlying cause: how hidden states drift apart across the depth of the network.

We'll attach forward hooks to every decoder layer to capture the hidden state tensor after each layer's residual addition. Then we compute the L2 norm of the difference between original and swapped at each `(layer, token_position)` point, giving us a 2D heatmap of where and how much the representations diverge.

In [ ]:
class HiddenStateCapture:
    """Registers forward hooks on decoder layers to capture hidden states."""

    def __init__(self):
        self.states: dict[int, torch.Tensor] = {}
        self._hooks = []

    def register(self, model):
        for i, layer in enumerate(model.model.layers):
            def _hook(module, inp, out, idx=i):
                hs = out[0] if isinstance(out, tuple) else out
                self.states[idx] = hs.detach().cpu().float()
            self._hooks.append(layer.register_forward_hook(_hook))

    def remove(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()

    def clear(self):
        self.states.clear()


def capture_hidden_states(model, input_ids: torch.Tensor) -> dict[int, torch.Tensor]:
    cap = HiddenStateCapture()
    cap.register(model)
    with torch.no_grad():
        model(input_ids.to(model.device))
    cap.remove()
    return cap.states


def compute_diff_matrix(
    states_orig: dict, states_swap: dict
) -> np.ndarray:
    """Returns (n_layers, seq_len) matrix of per-token L2 norms."""
    n_layers = len(states_orig)
    seq_len  = states_orig[0].shape[1]
    mat = np.zeros((n_layers, seq_len))
    for l in range(n_layers):
        diff = states_orig[l][0] - states_swap[l][0]   # [seq_len, hidden_dim]
        mat[l] = diff.norm(dim=-1).numpy()
    return mat


def plot_hidden_state_heatmap(
    diff_matrix: np.ndarray,
    title: str,
    token_labels: list[str] | None = None,
    ax=None,
    vmax=None,
):
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(
        diff_matrix, aspect="auto", cmap="magma", origin="lower",
        vmin=0, vmax=vmax or diff_matrix.max(),
    )
    plt.colorbar(im, ax=ax, label="L2 Norm of Difference")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Token Position", fontsize=10)
    ax.set_ylabel("Layer Index", fontsize=10)
    if token_labels and len(token_labels) <= 40:
        ax.set_xticks(range(len(token_labels)))
        ax.set_xticklabels(token_labels, rotation=45, ha="right", fontsize=7)
    if standalone:
        plt.tight_layout()
        plt.savefig(f"hidden_diff_{title.lower().replace(' ', '_')}.png", dpi=150, bbox_inches="tight")
        plt.show()

In [ ]:
ANALYSIS_PROMPT = "In 1969, NASA's Apollo 11 mission successfully landed the first humans on the Moon."
analysis_input_ids = tokenizer(ANALYSIS_PROMPT, return_tensors="pt")["input_ids"]
token_labels = [tokenizer.decode([t]) for t in analysis_input_ids[0].tolist()]

print(f"Analysing {len(token_labels)} tokens in bf16...")

states_bf16_orig = capture_hidden_states(model_bf16, analysis_input_ids)

orig_fwd = patch_model_rmsnorm(model_bf16, liger_kernel_rmsnorm)
states_bf16_swap = capture_hidden_states(model_bf16, analysis_input_ids)
unpatch_model_rmsnorm(model_bf16, orig_fwd)

diff_bf16 = compute_diff_matrix(states_bf16_orig, states_bf16_swap)
print(f"bf16 max L2 diff across all layers/positions: {diff_bf16.max():.4f}")
print(f"bf16 mean L2 diff: {diff_bf16.mean():.4f}")

In [ ]:
plot_hidden_state_heatmap(
    diff_bf16,
    title="Hidden State L2 Difference: Original vs Swapped Kernel (bf16)",
    token_labels=token_labels,
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

layer_mean_bf16 = diff_bf16.mean(axis=1)
layers = np.arange(len(layer_mean_bf16))

ax.plot(layers, layer_mean_bf16, label="bf16", color="#1976D2", linewidth=2, marker="o", markersize=4)
ax.fill_between(layers, layer_mean_bf16, alpha=0.15, color="#1976D2")

ax.set_xlabel("Layer Index", fontsize=12)
ax.set_ylabel("Mean L2 Norm of Hidden State Difference", fontsize=12)
ax.set_title(
    "Error Accumulation by Layer: Kernel-Swapped vs Original\n"
    "(mean across all token positions, bf16)",
    fontsize=13, fontweight="bold",
)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("layer_drift_bf16.png", dpi=150, bbox_inches="tight")
plt.show()

The layer-mean plot tells the story clearly: error does not stay constant across depth. It accumulates, sometimes monotonically, sometimes with spikes at layers that contain more normalization steps (e.g., pre/post-attention norms).

---

## Part 5: Does It Matter for Downstream Predictions? KL Divergence on MMLU-Pro

Hidden state drift is interesting; the practical question is whether it shows up in the model's output distribution. We measure this directly on **[MMLU-Pro](https://huggingface.co/datasets/TIGER-Lab/MMLU-Pro)**, a multiple-choice reasoning benchmark with up to 10 options per question.

For each of 100 randomly sampled questions, we:

1. Build a prompt of the form `Question: ...\nOptions:\nA. ...\n...\nAnswer:` and run a single forward pass through both the original and the kernel-swapped model.
2. Compute the **KL divergence** between the two next-token distributions: $D_{KL}(P_{\text{original}} \;\Vert\; P_{\text{swapped}})$.
3. Take the **argmax over the answer-letter tokens** (`A`-`J`) to get each model's predicted answer.

A swap that is "the same model" should produce near-zero KL and identical argmax answers on every question. We'll see how close to that ideal the swapped kernel really is, and on the questions where it isn't, we'll plot the shift in answer-letter probabilities.

In [ ]:
import random

ds = load_dataset("TIGER-Lab/MMLU-Pro", split="test")
N_QUESTIONS = 100
SEED = 42

random.seed(SEED)
sampled_idx = random.sample(range(len(ds)), N_QUESTIONS)
mmlu_pro_questions = [ds[i] for i in sampled_idx]

print(f"Sampled {len(mmlu_pro_questions)} questions from MMLU-Pro")
print("\nExample question:")
ex = mmlu_pro_questions[0]
print(f"  category : {ex['category']}")
print(f"  question : {ex['question']}")
print(f"  options  : {ex['options']}")
print(f"  answer   : {ex['answer']}")

In [ ]:
import torch.nn.functional as F

ANSWER_LETTERS = list(string.ascii_uppercase[:10])  # A..J

# Token IDs for " A", " B", ... " J" (with leading space, since the prompt ends with "Answer:")
LETTER_TOKEN_IDS = {
    L: tokenizer.encode(f" {L}", add_special_tokens=False)[0] for L in ANSWER_LETTERS
}


def format_mmlu_pro_prompt(question: dict) -> tuple[str, list[str]]:
    """Return prompt + the list of valid answer letters for this question."""
    options = question['options']
    letters = ANSWER_LETTERS[:len(options)]
    body = "\n".join(f"{L}. {opt}" for L, opt in zip(letters, options))
    prompt = (
        f"Question: {question['question']}\n"
        f"Options:\n{body}\n"
        f"Answer:"
    )
    return prompt, letters


@torch.no_grad()
def next_token_logits(model, prompt: str) -> torch.Tensor:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    logits = model(**inputs).logits[0, -1]
    return logits.float().cpu()


def kl_div(logits_p: torch.Tensor, logits_q: torch.Tensor) -> float:
    """KL(P || Q) in nats, where P=softmax(logits_p), Q=softmax(logits_q)."""
    log_p = F.log_softmax(logits_p, dim=-1)
    log_q = F.log_softmax(logits_q, dim=-1)
    p = log_p.exp()
    return (p * (log_p - log_q)).sum().item()


def letter_argmax(logits: torch.Tensor, letters: list[str]) -> tuple[str, dict[str, float]]:
    """Argmax restricted to valid answer letters. Returns (letter, prob-dict over letters)."""
    ids = [LETTER_TOKEN_IDS[L] for L in letters]
    sub_logits = logits[ids]
    probs = F.softmax(sub_logits, dim=-1).tolist()
    prob_map = dict(zip(letters, probs))
    pick = letters[int(np.argmax(probs))]
    return pick, prob_map

In [ ]:
results = []
for q in tqdm(mmlu_pro_questions, desc="MMLU-Pro: original kernel"):
    prompt, letters = format_mmlu_pro_prompt(q)
    logits = next_token_logits(model_bf16, prompt)
    pick, probs = letter_argmax(logits, letters)
    results.append({
        'prompt': prompt,
        'letters': letters,
        'gold': q['answer'],
        'orig_logits': logits,
        'orig_pick': pick,
        'orig_probs': probs,
    })

orig_fwd = patch_model_rmsnorm(model_bf16, liger_kernel_rmsnorm)
for r in tqdm(results, desc="MMLU-Pro: swapped kernel"):
    logits = next_token_logits(model_bf16, r['prompt'])
    pick, probs = letter_argmax(logits, r['letters'])
    r['swap_logits'] = logits
    r['swap_pick'] = pick
    r['swap_probs'] = probs
unpatch_model_rmsnorm(model_bf16, orig_fwd)

for r in results:
    r['kl'] = kl_div(r['orig_logits'], r['swap_logits'])

kl_values = np.array([r['kl'] for r in results])
print(f"\nKL(P_orig || P_swap) over {len(results)} MMLU-Pro questions:")
print(f"  mean   : {kl_values.mean():.4e}")
print(f"  median : {np.median(kl_values):.4e}")
print(f"  max    : {kl_values.max():.4e}")
print(f"  min    : {kl_values.min():.4e}")

n_flipped = sum(1 for r in results if r['orig_pick'] != r['swap_pick'])
print(f"\nArgmax answer flipped: {n_flipped} / {len(results)} questions")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(kl_values, bins=30, color='#1565C0', edgecolor='white', linewidth=0.5)
ax.axvline(np.mean(kl_values), color='#D32F2F', linestyle='--', linewidth=1.5,
           label=f"mean = {kl_values.mean():.2e}")
ax.set_xlabel("KL(P_original || P_swapped)  (nats)", fontsize=11)
ax.set_ylabel("Number of questions", fontsize=11)
ax.set_title("KL Divergence on 100 MMLU-Pro Questions\n(Qwen3-0.6B, bf16)",
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
flipped_mask = np.array([r['orig_pick'] != r['swap_pick'] for r in results])
ax.scatter(np.where(~flipped_mask)[0], kl_values[~flipped_mask],
           s=24, color='#1565C0', alpha=0.7, label='argmax unchanged')
ax.scatter(np.where(flipped_mask)[0], kl_values[flipped_mask],
           s=44, color='#D32F2F', alpha=0.95, edgecolor='black',
           label=f'argmax flipped (n={int(flipped_mask.sum())})')
ax.set_yscale('log')
ax.set_xlabel("Question index (0..99)", fontsize=11)
ax.set_ylabel("KL divergence  (nats, log scale)", fontsize=11)
ax.set_title("Per-Question KL: Flipped vs Unchanged Answers", fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('mmlu_pro_kl.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
unchanged = [r for r in results if r['orig_pick'] == r['swap_pick']]
print(f"Questions with argmax answer unchanged: {len(unchanged)}")

if unchanged:
    pick_logit_deltas = np.array([
        (r['swap_logits'][LETTER_TOKEN_IDS[r['orig_pick']]]
         - r['orig_logits'][LETTER_TOKEN_IDS[r['orig_pick']]]).item()
        for r in unchanged
    ])
    letter_logit_deltas = np.concatenate([
        (r['swap_logits'][[LETTER_TOKEN_IDS[L] for L in r['letters']]]
         - r['orig_logits'][[LETTER_TOKEN_IDS[L] for L in r['letters']]]).numpy()
        for r in unchanged
    ])

    print(f"\nΔlogit at the (shared) argmax letter, n={len(pick_logit_deltas)}:")
    print(f"  mean    : {pick_logit_deltas.mean():+.4e}")
    print(f"  median  : {np.median(pick_logit_deltas):+.4e}")
    print(f"  std     : {pick_logit_deltas.std():.4e}")
    print(f"  min     : {pick_logit_deltas.min():+.4e}")
    print(f"  max     : {pick_logit_deltas.max():+.4e}")
    print(f"  mean|·| : {np.abs(pick_logit_deltas).mean():.4e}")

    print(f"\nΔlogit across ALL answer-letter tokens (unchanged questions), n={len(letter_logit_deltas)}:")
    print(f"  mean    : {letter_logit_deltas.mean():+.4e}")
    print(f"  std     : {letter_logit_deltas.std():.4e}")
    print(f"  min     : {letter_logit_deltas.min():+.4e}")
    print(f"  max     : {letter_logit_deltas.max():+.4e}")
    print(f"  mean|·| : {np.abs(letter_logit_deltas).mean():.4e}")

flipped = [r for r in results if r['orig_pick'] != r['swap_pick']]
print(f"\nQuestions with argmax answer flipped: {len(flipped)}")

if flipped:
    rows = []
    for r in flipped:
        a, b = r['orig_pick'], r['swap_pick']
        rows.append({
            'orig_pick': a,
            'swap_pick': b,
            'delta_orig': r['swap_probs'][a] - r['orig_probs'][a],  # change in P(original answer)
            'delta_swap': r['swap_probs'][b] - r['orig_probs'][b],  # change in P(modified answer)
        })

    n = len(rows)
    x = np.arange(n)
    w = 0.4

    fig, ax = plt.subplots(figsize=(max(8, 0.9 * n), 5))

    ax.bar(x - w/2, [r['delta_orig'] for r in rows], w,
           color='#1565C0', edgecolor='black', linewidth=0.6,
           label='ΔP(original answer)  =  P_swap - P_orig')
    ax.bar(x + w/2, [r['delta_swap'] for r in rows], w,
           color='#EF6C00', edgecolor='black', linewidth=0.6,
           label='ΔP(modified answer)  =  P_swap - P_orig')

    ax.axhline(0, color='black', linewidth=0.8)
    labels = [f"{r['orig_pick']}->{r['swap_pick']}" for r in rows]
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_xlabel("Flipped questions  (orig pick -> swap pick)", fontsize=11)
    ax.set_ylabel("Change in probability (renormalised over answer letters)", fontsize=11)
    ax.set_title(
        "Probability Change on Argmax-Flipped Questions\n(Qwen3-0.6B, bf16)",
        fontsize=12, fontweight='bold',
    )
    ax.legend(fontsize=10, loc='best')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('mmlu_pro_prob_change.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No flipped answers: argmax stable under kernel swap.")

---

## Takeaways

### 1. Kernel-level correctness is necessary but not sufficient
The Liger RMSNorm kernel passes atol/rtol correctness checks in float32 and bf16. By every standard unit-test criterion, it is a valid drop-in replacement. Yet the swapped model produces a measurably different output distribution on MMLU-Pro, and on a non-trivial fraction of questions it picks a different answer than the original.

### 2. KL divergence is a sensitive, single-pass diagnostic
Single-pass argmax benchmarks like MMLU often show no difference between original and swapped kernels: the noise sits below the threshold that flips the top choice. Looking at KL divergence over the full next-token distribution exposes the drift directly, without needing long generations.

### 3. Errors accumulate non-linearly across layers
The layer-by-layer hidden state drift plot shows divergence growing with depth, with sharper jumps at layers that apply normalization multiple times. The per-question KL on MMLU-Pro is the downstream consequence of that accumulation.

### 4. The right validation protocol for kernel swapping
If you're swapping kernels in a production model:
1. Pass standard kernel unit tests (atol/rtol) ✓, necessary but not sufficient
2. Run end-to-end generation comparison with greedy decoding
3. Sample a benchmark and compute the per-question KL divergence between original and swapped output distributions, at the exact dtype you'll use in production
4. Monitor hidden-state drift across layers as a diagnostic

---

*Code for this post is available in the companion repository. Liger-Kernel is available via [`pip install liger-kernel`](https://pypi.org/project/liger-kernel/). MMLU-Pro is available at [`TIGER-Lab/MMLU-Pro`](https://huggingface.co/datasets/TIGER-Lab/MMLU-Pro).*
